# 00 — Carga y validación de datos

**Grupo 10 — Minería de Datos**

Este notebook carga los 3 datasets del MINEDUC, valida su integridad,
realiza el join por MRUN y guarda los datasets procesados en `data/processed/`.

Solo necesita ejecutarse una vez. Los demás notebooks cargan desde `data/processed/`.

## Datasets
| ID | Archivo | Descripción | Filas esperadas |
|----|---------|--------------|-----------------|
| A | `A_puntajes.csv` | Inscritos PAES 2025 + puntajes | ~313.759 |
| B | `B_socioeconomico.csv` | Domicilio y datos socioeconómicos | ~313.759 |
| C | `C_matricula.csv` | Matrícula Ed. Superior 2025 | ~1.455.639 |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─── Rutas ──────────────────────────────────────────────────────────────────
# Ajustar si los CSVs están en otra ubicación (Drive, Colab, etc.)
DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

print('Rutas configuradas:')
print(f'  RAW:       {DATA_RAW.resolve()}')
print(f'  PROCESSED: {DATA_PROC.resolve()}')

## 1. Carga de datasets

In [ ]:
# ─── Dataset A: Puntajes PAES ────────────────────────────────────────────────
# Columnas que usaremos — evitar cargar todo el CSV (son ~100 columnas)
COLS_A = [
    'MRUN',
    'RAMA_EDUCACIONAL',   # H1-H4 humanista, T1-T5 técnico-profesional
    'DEPENDENCIA',        # 1-6 tipo de colegio
    'CODIGO_REGION_EGRESO',
    'ANYO_DE_EGRESO',
    'PTJE_NEM',
    'PTJE_RANKING',
    'CLEC_MAX',
    'MATE1_MAX',
    'MATE2_MAX',
    'HCSOC_MAX',
    'CIEN_MAX',
    # Distribución de respuestas (para análisis avanzado)
    'CORRECTAS_REG_CL', 'ERRADAS_REG_CL', 'OMITIDAS_REG_CL',
    'CORRECTAS_REG_M1', 'ERRADA_REG_M1', 'OMITIDAS_REG_M1',
]

print('Cargando Dataset A (puntajes)...')
df_A = pd.read_csv(
    DATA_RAW / 'A_puntajes.csv',
    usecols=lambda c: c in COLS_A,
    encoding='utf-8',
    low_memory=False
)
print(f'  Filas: {len(df_A):,} | Columnas: {df_A.shape[1]}')
df_A.head(3)

In [ ]:
# ─── Dataset B: Socioeconómico ───────────────────────────────────────────────
COLS_B = [
    'MRUN',
    'CODIGO_REGION_DOMICILIO',
    'CODIGO_PROVINCIA_DOMICILIO',
    'CODIGO_COMUNA_DOMICILIO',
    'NOMBRE_COMUNA_DOMICILIO',
    'SEXO',
    'INGRESO_PERCAPITA_GRUPO_FA',   # decil 1-10 (autodeclarado)
    'RAZON_PRINCIPAL_PAES',          # motivación del estudiante
]

print('Cargando Dataset B (socioeconómico)...')
df_B = pd.read_csv(
    DATA_RAW / 'B_socioeconomico.csv',
    usecols=lambda c: c in COLS_B,
    encoding='utf-8',
    low_memory=False
)
print(f'  Filas: {len(df_B):,} | Columnas: {df_B.shape[1]}')
df_B.head(3)

In [ ]:
# ─── Dataset C: Matrícula ────────────────────────────────────────────────────
COLS_C = [
    'MRUN',
    'CAT_PERIODO',          # año (filtrar solo 2025)
    'TIPO_INST_1',          # CFT / IP / Universidad
    'TIPO_INST_2',          # CFT / IP / U.CRUCH / U.Privada  ← variable objetivo P3
    'NOMB_INST',
    'NOMB_CARRERA',
    'AREA_CONOCIMIENTO',    # área UNESCO de la carrera
    'REGION_SEDE',          # región donde está la institución ← para P4 (migra)
    'FORMA_DE_INGRESO',     # PAES, PACE, inclusión, etc.
    'VALOR_ARANCEL',
    'ACREDITADA_INST',
    'NIVEL_GLOBAL',         # pregrado / postgrado / postítulo
    'NIVEL_CARRERA_2',      # carrera técnica / profesional
]

print('Cargando Dataset C (matrícula)...')
df_C = pd.read_csv(
    DATA_RAW / 'C_matricula.csv',
    usecols=lambda c: c in COLS_C,
    encoding='utf-8',
    low_memory=False
)

# Filtrar solo matrículas de 2025
if 'CAT_PERIODO' in df_C.columns:
    df_C = df_C[df_C['CAT_PERIODO'] == 2025]
    
print(f'  Filas: {len(df_C):,} | Columnas: {df_C.shape[1]}')
df_C.head(3)

## 2. Validación básica

In [ ]:
def validar_dataset(df, nombre):
    print(f'\n{'='*50}')
    print(f'Dataset {nombre}: {len(df):,} filas x {df.shape[1]} columnas')
    print(f'MRUNs únicos: {df["MRUN"].nunique():,}')
    print(f'MRUNs duplicados: {df["MRUN"].duplicated().sum():,}')
    
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(f'Columnas con nulos:')
        for col, n in nulos.items():
            print(f'  {col}: {n:,} ({n/len(df)*100:.1f}%)')
    else:
        print('Sin valores nulos')

validar_dataset(df_A, 'A (puntajes)')
validar_dataset(df_B, 'B (socioeconómico)')
validar_dataset(df_C, 'C (matrícula 2025)')

## 3. Join A + B → base para P1 y P2

In [ ]:
print('Join A + B por MRUN...')
df_base = df_A.merge(df_B, on='MRUN', how='inner')
print(f'Antes del join: A={len(df_A):,} | B={len(df_B):,}')
print(f'Después del join: {len(df_base):,} filas')
print(f'Pérdida: {len(df_A) - len(df_base):,} registros sin match')

In [ ]:
# ─── Filtro 1: Excluir quienes no rindieron la PAES ─────────────────────────
# Puntaje 0 significa que se inscribió pero no rindió
n_antes = len(df_base)
df_base = df_base[df_base['CLEC_MAX'] > 0]
print(f'Filtro puntaje > 0: eliminados {n_antes - len(df_base):,} registros')

# ─── Filtro 2: Excluir decil inválido (99 = prefiero no responder) ───────────
n_antes = len(df_base)
df_base = df_base[df_base['INGRESO_PERCAPITA_GRUPO_FA'] != 99]
print(f'Filtro decil válido: eliminados {n_antes - len(df_base):,} registros')

print(f'\nDataset base final: {len(df_base):,} estudiantes válidos')

## 4. Join base + C → dataset completo para P3 y P4

In [ ]:
print('Join base + C por MRUN...')
df_completo = df_base.merge(df_C, on='MRUN', how='inner')
print(f'Base: {len(df_base):,} | Con matrícula: {len(df_completo):,}')
print(f'Estudiantes que NO se matricularon: {len(df_base) - len(df_completo):,}')
print(f'Tasa de matriculación: {len(df_completo)/len(df_base)*100:.1f}%')

## 5. Construcción de variables derivadas

In [ ]:
# ─── Variable: es_tecnico ────────────────────────────────────────────────────
# 1 si el estudiante viene de colegio técnico-profesional, 0 si humanista
df_base['es_tecnico'] = df_base['RAMA_EDUCACIONAL'].str.startswith('T').astype(int)
df_completo['es_tecnico'] = df_completo['RAMA_EDUCACIONAL'].str.startswith('T').astype(int)

print('Distribución rama educacional:')
print(df_base['RAMA_EDUCACIONAL'].value_counts())
print(f'\nTécnico-profesionales: {df_base["es_tecnico"].sum():,} ({df_base["es_tecnico"].mean()*100:.1f}%)')

In [ ]:
# ─── Variable: migra ─────────────────────────────────────────────────────────
# Mapeo de código de región a nombre (según Anexo I del dataset B)
REGION_MAP = {
    1: 'Región de Tarapacá',
    2: 'Región de Antofagasta',
    3: 'Región de Atacama',
    4: 'Región de Coquimbo',
    5: 'Región de Valparaíso',
    6: "Región del Libertador General Bernardo O'Higgins",
    7: 'Región del Maule',
    8: 'Región del Biobío',
    9: 'Región de La Araucanía',
    10: 'Región de Los Lagos',
    11: 'Región Aysén del General Carlos Ibáñez del Campo',
    12: 'Región de Magallanes y de la Antártica Chilena',
    13: 'Región Metropolitana de Santiago',
    14: 'Región de Los Ríos',
    15: 'Región de Arica y Parinacota',
    16: 'Región de Ñuble',
}

df_completo['region_domicilio_nombre'] = df_completo['CODIGO_REGION_DOMICILIO'].map(REGION_MAP)

# migra = 1 si la región de la institución es distinta a la de domicilio
# Nota: REGION_SEDE viene como nombre en el Dataset C
df_completo['migra'] = (
    df_completo['region_domicilio_nombre'].str.lower().str.strip() != 
    df_completo['REGION_SEDE'].str.lower().str.strip()
).astype(int)

print('Distribución variable migra:')
print(df_completo['migra'].value_counts())
print(f'\nTasa de migración: {df_completo["migra"].mean()*100:.1f}%')

## 6. Verificación de calidad — variables clave

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribución de variables clave — Dataset base', fontsize=14)

# Puntajes
df_base['CLEC_MAX'].hist(bins=50, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Puntaje CLEC (lenguaje)')

df_base['MATE1_MAX'].hist(bins=50, ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Puntaje MATE1 (matemática)')

df_base['PTJE_NEM'].hist(bins=50, ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Puntaje NEM')

# Variables socioeconómicas
df_base['INGRESO_PERCAPITA_GRUPO_FA'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,0], color='coral')
axes[1,0].set_title('Decil de ingreso familiar')
axes[1,0].set_xlabel('Decil')

df_base['DEPENDENCIA'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,1], color='coral')
axes[1,1].set_title('Dependencia del colegio')
labels = ['Corp.Mun', 'Municipal', 'Part.Subv', 'Part.Pag', 'Corp.Del', 'SLE']
axes[1,1].set_xticklabels(labels, rotation=30)

df_base['es_tecnico'].value_counts().plot(
    kind='bar', ax=axes[1,2], color='coral')
axes[1,2].set_title('Rama: Técnico vs Humanista')
axes[1,2].set_xticklabels(['Humanista', 'Técnico'], rotation=0)

plt.tight_layout()
plt.savefig('../data/processed/distribucion_variables.png', dpi=100, bbox_inches='tight')
plt.show()
print('Gráfico guardado en data/processed/')

In [ ]:
# ─── Verificar colinealidad NEM / Ranking ────────────────────────────────────
corr = df_base[['PTJE_NEM', 'PTJE_RANKING', 'CLEC_MAX', 'MATE1_MAX']].corr()
print('Matriz de correlación (variables numéricas principales):')
print(corr.round(3))

corr_nem_ranking = corr.loc['PTJE_NEM', 'PTJE_RANKING']
print(f'\nCorrelación NEM ↔ Ranking: {corr_nem_ranking:.3f}')
if corr_nem_ranking > 0.85:
    print('⚠️  Alta correlación — considerar eliminar PTJE_NEM del clustering')
else:
    print('✅  Correlación aceptable — se pueden usar ambas')

## 7. Guardar datasets procesados

Usamos **Parquet** en lugar de CSV porque:
- Ocupa ~5x menos espacio
- Carga ~10x más rápido
- Conserva los tipos de datos

In [ ]:
# Guardar en formato Parquet
df_base.to_parquet(DATA_PROC / 'df_base.parquet', index=False)
df_completo.to_parquet(DATA_PROC / 'df_completo.parquet', index=False)

print('Archivos guardados en data/processed/:')
print(f'  df_base.parquet     → {len(df_base):,} filas (P1, P2)')
print(f'  df_completo.parquet → {len(df_completo):,} filas (P3, P4)')

# Verificar tamaños
import os
for f in ['df_base.parquet', 'df_completo.parquet']:
    size = os.path.getsize(DATA_PROC / f) / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# ─── Crear muestra pequeña para el repo (commit en git) ──────────────────────
sample = df_base.sample(1000, random_state=42)
sample.to_csv('../data/samples/sample_1000.csv', index=False)
print('Muestra de 1000 filas guardada en data/samples/sample_1000.csv')
print('Este archivo SÍ va al repositorio git.')

## 8. Resumen final

| Dataset | Filas | Para |  
|---------|-------|------|
| `df_base` | ~194k | P1 Clustering, P2 Regresión |
| `df_completo` | ~100-120k | P3 Clasificación tipo inst., P4 Migración |

**Notas importantes:**
- El decil de ingreso es **autodeclarado** → sesgo de reporte posible
- Los estudiantes que no se matricularon (~74k) quedan fuera del análisis de P3/P4
- La variable `migra` compara nombres de región — verificar consistencia de strings

**Siguiente paso:** ejecutar `01_preprocesamiento.ipynb`